# SI26-Week2-Humna
**Project:** Urdu OCR — Code Saviours ML/AI Internship (Batch SI-26)
**Week 2:** Image Preprocessing + Testing Existing OCR Tools

This notebook:
1. Loads your raw Urdu text images (path set by `RAW_DIR` — see Step 2b)
2. Preprocesses them (grayscale → aspect-preserving resize/pad → denoise → binarise) and saves the results to `PROCESSED_DIR`
3. Runs baseline Tesseract OCR on the processed images to see how well an *existing* OCR tool handles Urdu
4. Documents a gap analysis explaining why Tesseract struggles, motivating why this project needs a custom model

**Note on this version:** an earlier attempt at this notebook used a fixed global threshold (`cv2.threshold(img, 127, 255, ...)`) and a hard resize to `(512, 128)`. On real photographed pages with uneven lighting, a single fixed brightness cutoff turns shadowed regions into speckled noise ("dotted" output), and forcing every image into the same box regardless of its original aspect ratio stretches/warps the Urdu characters. This version fixes both: it uses **Otsu's method** (which recalculates the best threshold per image automatically) and an **aspect-ratio-preserving resize with padding**.


## Part A — Preprocess Your Images

### Step 2 — Install Libraries

In [7]:
!pip install opencv-python-headless pillow matplotlib

import cv2
import numpy as np
from PIL import Image
import os
import glob
import matplotlib.pyplot as plt

print('Libraries loaded successfully!')



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
Libraries loaded successfully!


### Step 2b — Point This Notebook at Your Data (GitHub Codespaces)

You're running in a Codespace, so your repo is already on this machine — nothing to upload.
But the notebook's working directory is wherever this `.ipynb` file sits in your repo, which is
probably a **different folder** than your Week 1 data (e.g. this notebook lives in `SI26-Week2/`
while your images are in `SI26-Week1/data/raw/`). A relative path like `data/raw` only works if
`data/raw` is directly next to this notebook file.

Run the auto-locate cell below first — it searches your whole workspace for a folder literally
named `raw` containing images and prints the exact path(s) it finds. Copy the correct one into
`RAW_DIR` in the cell after that.

(Manual alternative: in the VS Code Explorer sidebar on the left, find your data folder, right-click
any image inside `raw/` → **Copy Relative Path**, then use that folder — without the filename — as `RAW_DIR`.)

In [8]:
import os

def find_raw_folders(start_path, max_depth=6):
    found = []
    start_path = os.path.abspath(start_path)
    if not os.path.exists(start_path):
        return found
    base_depth = start_path.rstrip(os.sep).count(os.sep)
    for root, dirs, files in os.walk(start_path):
        depth = root.rstrip(os.sep).count(os.sep) - base_depth
        if depth > max_depth:
            dirs[:] = []
            continue
        # skip irrelevant/heavy folders
        dirs[:] = [d for d in dirs if d not in ('.git', 'node_modules', '__pycache__', '.venv', 'venv')]
        if os.path.basename(root).lower() == 'raw':
            img_count = sum(1 for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png')))
            found.append((root, img_count))
    return found

print('Notebook working directory:', os.getcwd())
print()

# Search the whole Codespace workspace (holds your full repo) plus nearby folders
search_roots = ['/workspaces', os.path.join(os.getcwd(), '..', '..'), os.getcwd()]
candidates = []
for sr in search_roots:
    candidates.extend(find_raw_folders(sr))
candidates = sorted(set(candidates))

if candidates:
    print("Found candidate 'raw' folders (path, image count):")
    for path, count in candidates:
        print(f'  {path}   ({count} images)')
    print()
    print("Copy the correct path above into RAW_DIR in the next cell.")
else:
    print("No folder literally named 'raw' with images found automatically.")
    print("Use the Explorer sidebar in VS Code: find your data folder, right-click an image ->")
    print("'Copy Relative Path', and use that folder (minus the filename) as RAW_DIR below.")


Notebook working directory: /workspaces/Urdu-OCR-Project-Code-Saviours-SI-26-Humna-Imran/SI26-Week2

Found candidate 'raw' folders (path, image count):
  /workspaces/Urdu-OCR-Project-Code-Saviours-SI-26-Humna-Imran/SI26-Week1/data/raw   (0 images)

Copy the correct path above into RAW_DIR in the next cell.


In [9]:
# ---- SET THIS to wherever your raw images actually end up ----
RAW_DIR = 'data/raw'            # e.g. 'SI26-Week1/data/raw' or '/content/drive/MyDrive/.../data/raw'
PROCESSED_DIR = 'data/processed'

import os
os.makedirs(PROCESSED_DIR, exist_ok=True)
print(f'RAW_DIR set to: {RAW_DIR}')
print(f'PROCESSED_DIR set to: {PROCESSED_DIR}')


RAW_DIR set to: data/raw
PROCESSED_DIR set to: data/processed


### Step 3 — Preprocessing Function (fixed version)

Changes from the original handout version, and why:

| Old (caused dotted output) | New | Why |
|---|---|---|
| `cv2.resize(gray, (512, 128))` — forces every image into the same box | Resize preserving aspect ratio, then pad onto a fixed white canvas | Hard resize stretches/squashes characters — bad for joined Urdu script |
| `cv2.threshold(img, 127, 255, THRESH_BINARY)` — one fixed brightness cutoff for the whole image | `cv2.threshold(img, 0, 255, THRESH_BINARY + THRESH_OTSU)` | Otsu recalculates the optimal cutoff per image, so uneven lighting/shadows don't turn into speckled dots |
| Denoise strength `h=10` | Denoise strength `h=7` | Slightly gentler — strong denoising can erode thin Urdu strokes and diacritic dots before they even reach the threshold step |


In [10]:
def preprocess_image(image_path, save_path, target_size=(800, 160)):
    """
    Preprocess a single image for OCR:
    1. Grayscale
    2. Resize preserving aspect ratio, padded onto a fixed white canvas (no stretching)
    3. Light denoising
    4. Otsu binarisation (auto threshold per image -> no speckling on uneven lighting)
    """
    img = cv2.imread(image_path)
    if img is None:
        print(f'Could not load: {image_path}')
        return None

    # Step 1: Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Step 2: Resize preserving aspect ratio, then pad onto a white canvas
    target_w, target_h = target_size
    h, w = gray.shape
    scale = min(target_w / w, target_h / h)
    new_w, new_h = max(1, int(w * scale)), max(1, int(h * scale))
    resized = cv2.resize(gray, (new_w, new_h), interpolation=cv2.INTER_CUBIC)

    canvas = np.full((target_h, target_w), 255, dtype=np.uint8)
    y_off = (target_h - new_h) // 2
    x_off = (target_w - new_w) // 2
    canvas[y_off:y_off + new_h, x_off:x_off + new_w] = resized

    # Step 3: Light denoise (too strong erodes thin strokes/dots in Urdu script)
    denoised = cv2.fastNlMeansDenoising(canvas, h=7)

    # Step 4: Otsu binarisation - auto threshold per image, fixes 'dotted' output
    _, binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    cv2.imwrite(save_path, binary)
    return binary

print('Preprocessing function ready!')


Preprocessing function ready!


In [11]:
# Find all images in RAW_DIR
all_images = glob.glob(f'{RAW_DIR}/**/*.jpg', recursive=True)
all_images += glob.glob(f'{RAW_DIR}/**/*.jpeg', recursive=True)
all_images += glob.glob(f'{RAW_DIR}/**/*.png', recursive=True)

print(f'Found {len(all_images)} images to process in {RAW_DIR}')

processed_count = 0
for img_path in all_images:
    filename = os.path.splitext(os.path.basename(img_path))[0] + '.png'
    save_path = os.path.join(PROCESSED_DIR, filename)
    result = preprocess_image(img_path, save_path)
    if result is not None:
        processed_count += 1

print(f'Done! Processed {processed_count} images')
print(f'Check {PROCESSED_DIR} folder')


Found 0 images to process in data/raw
Done! Processed 0 images
Check data/processed folder


### Quick visual sanity check
Before moving to OCR, eyeball a few processed images to confirm the dotted-noise issue is gone (should be clean black text on white, no speckling).

In [12]:
sample_paths = glob.glob(f'{PROCESSED_DIR}/*.png')[:4]

if len(sample_paths) == 0:
    print(f"No processed images found in {PROCESSED_DIR}.")
    print("Scroll up and check the 'Found X images to process' / 'Done! Processed X images' messages")
    print(f"from the preprocessing cell. If 'Found 0 images', RAW_DIR ('{RAW_DIR}') is wrong for")
    print("this environment - rerun the diagnostic cell above, fix RAW_DIR, then rerun that cell.")
else:
    fig, axes = plt.subplots(1, len(sample_paths), figsize=(16, 4))
    if len(sample_paths) == 1:
        axes = [axes]
    for ax, p in zip(axes, sample_paths):
        ax.imshow(plt.imread(p), cmap='gray')
        ax.set_title(os.path.basename(p), fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


No processed images found in data/processed.
Scroll up and check the 'Found X images to process' / 'Done! Processed X images' messages
from the preprocessing cell. If 'Found 0 images', RAW_DIR ('data/raw') is wrong for
this environment - rerun the diagnostic cell above, fix RAW_DIR, then rerun that cell.


## Part B — Test Tesseract OCR on Your Urdu Images

**Note for Codespaces:** if `!apt-get install` below fails with a permissions error,
run `!sudo apt-get update && sudo apt-get install -y tesseract-ocr tesseract-ocr-urd` instead
(Codespaces containers usually run as a non-root user, unlike Colab).

In [13]:
!apt-get install -y tesseract-ocr tesseract-ocr-urd || sudo apt-get install -y tesseract-ocr tesseract-ocr-urd
!pip install pytesseract

import pytesseract
from PIL import Image

# Test on 5 of your processed images
test_images = list(glob.glob(f'{PROCESSED_DIR}/*.png'))[:5]

print('=== Tesseract Results on Urdu Images ===')
print()
for img_path in test_images:
    img = Image.open(img_path)
    # 'urd' tells Tesseract to use the Urdu language model
    result = pytesseract.image_to_string(img, lang='urd')
    print(f'Image: {img_path}')
    print(f'Tesseract output: {result}')
    print('---')


E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package tesseract-ocr
E: Unable to locate package tesseract-ocr-urd

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
=== Tesseract Results on Urdu Images ===



### Step 4 — Gap Analysis

For **each of the 5 test images above**, fill in the table below (replace the placeholders with what you actually observe):

| Image | Actual Urdu text | Tesseract output | What went wrong |
|---|---|---|---|
| image_1.png | *(type the real text here)* | *(paste Tesseract's output here)* | *(wrong characters / missing words / gibberish / merged letters — describe it)* |
| image_2.png | | | |
| image_3.png | | | |
| image_4.png | | | |
| image_5.png | | | |

**Summary paragraph** (start with this sentence and finish it based on your actual results):

> "Tesseract fails on Urdu because ..."

*(Think about: Urdu is written right-to-left in a cursive, context-dependent script where a letter's shape changes depending on its position in a word (Nastaliq style); Tesseract's Urdu model is trained mostly on cleaner, printed Naskh-style text; diacritics/dots are easy to lose during binarisation; word segmentation assumes left-to-right Latin-style spacing. Adjust this to match what you actually saw in your 5 images.)*

Copy this analysis into your GitHub README under a new section called **Why We Need a Better Model**.
